In [3]:
# -*- coding: utf-8 -*-
# ============================================================
#  save_initial_path_data.py
#
#  只初始化路径（C1 -> C2 线性插值, N=20），做一次 MD 评估，
#  计算与绘图 cell 完全一致的初始指标，并单独保存。
#  不做任何优化 / NEB 更新。
# ============================================================

import os
import glob
import time
import subprocess

import numpy as np
import torch

# ============================================================
# GROMACS / PLUMED 设置（与主脚本保持一致）
# ============================================================
GMX_EXECUTABLE = "/usr/local/gromacs/bin/gmx"
PLUMED_KERNEL_PATH = "/Users/wuzhiyou/plumed/lib/libplumedKernel.dylib"
PLUMED_BIN = "/Users/wuzhiyou/plumed/bin/plumed"

GROMACS_QUIET = True

GMX_BIN_DIR = os.path.dirname(GMX_EXECUTABLE)
os.environ["PATH"] = GMX_BIN_DIR + os.pathsep + os.environ.get("PATH", "")
os.environ["PLUMED_KERNEL"] = PLUMED_KERNEL_PATH

deg_to_rad = np.pi / 180.0

# ============================================================
# 固定几何与保存路径
# ============================================================
C1 = np.array([-2.60866606, 2.77661136], dtype=float)
C2 = np.array([-1.33047160, 0.99118795], dtype=float)

N = 20

base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

# 初始评估使用的 discard fraction（与主脚本 gradient_md_use 默认一致）
INIT_DISCARD_FRAC = 0.05

# 输出文件
OUTPUT_NPZ = os.path.join(base_path, "MFEP_INITIAL_PATH_common.npz")


# ============================================================
# shell 命令
# ============================================================
def run_shell_command(command, quiet=None):
    if quiet is None:
        quiet = GROMACS_QUIET
    kwargs = {"shell": True, "check": True}
    if quiet:
        kwargs["stdout"] = subprocess.DEVNULL
        kwargs["stderr"] = subprocess.STDOUT
    return subprocess.run(command, **kwargs)


def compute_discard_n(n_total, discard_frac):
    if discard_frac is None:
        return 0
    if not (0.0 <= discard_frac < 1.0):
        raise ValueError(f"discard_frac must satisfy 0 <= f < 1, got {discard_frac}")
    discard_n = int(np.floor(n_total * discard_frac))
    if discard_n >= n_total and n_total > 0:
        raise ValueError(f"discard_frac removes all frames: {discard_frac}")
    return discard_n


# ============================================================
# MD 相关（从主脚本复制，仅保留初始评估所需部分）
# ============================================================
def restrained_simulation(grid_point):
    phi_deg, psi_deg, phi, psi = grid_point
    dir_name = f"phi_{phi_deg}_psi_{psi_deg}"
    os.makedirs(dir_name, exist_ok=True)

    plumed_content_initial = f"""
MOLINFO STRUCTURE=./ala2.pdb

phi: TORSION ATOMS=@phi-2
psi: TORSION ATOMS=@psi-2
restraint: RESTRAINT ARG=phi,psi AT={phi},{psi} KAPPA=100.0,100.0

PRINT ARG=phi,psi,restraint.bias FILE={dir_name}/COLVAR_initial STRIDE=100
"""

    plumed_content = f"""
MOLINFO STRUCTURE=./ala2.pdb

phi: TORSION ATOMS=@phi-2
psi: TORSION ATOMS=@psi-2
restraint: RESTRAINT ARG=phi,psi AT={phi},{psi} KAPPA=1000.0,1000.0

DUMPPROJECTIONS ARG=phi,psi FILE={dir_name}/PROJ STRIDE=20
PRINT ARG=phi,psi,restraint.bias FILE={dir_name}/COLVAR_restrained STRIDE=20
"""

    with open(os.path.join(dir_name, "plumed.dat"), "w", encoding="utf-8") as f:
        f.write(plumed_content)
    with open(os.path.join(dir_name, "plumed_initial.dat"), "w", encoding="utf-8") as f:
        f.write(plumed_content_initial)

    verbose_flag = "" if GROMACS_QUIET else " -v"

    run_shell_command(
        f'"{GMX_EXECUTABLE}" mdrun -s nvt.tpr '
        f"-plumed {dir_name}/plumed_initial.dat "
        f"-nb cpu -pme cpu -ntmpi 1 -deffnm {dir_name}/initial{verbose_flag}"
    )
    run_shell_command(
        f'"{GMX_EXECUTABLE}" grompp -f nvt.mdp '
        f"-c {dir_name}/initial.gro -r {dir_name}/initial.gro "
        f"-p topol.top -o {dir_name}/restrained.tpr"
    )
    run_shell_command(
        f'"{GMX_EXECUTABLE}" mdrun -s {dir_name}/restrained.tpr '
        f"-plumed {dir_name}/plumed.dat "
        f"-nb cpu -pme cpu -ntmpi 1 -deffnm {dir_name}/restrained{verbose_flag}"
    )


def metric_from_proj(proj_path, tmin=None, tmax=None, discard_frac=0.0):
    arr = np.loadtxt(proj_path, comments="#")
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    t = arr[:, 0]
    g11, g12, g21, g22 = arr[:, 1:5].T

    mask = np.ones_like(t, dtype=bool)
    if tmin is not None:
        mask &= t >= tmin
    if tmax is not None:
        mask &= t <= tmax
    t, g11, g12, g21, g22 = t[mask], g11[mask], g12[mask], g21[mask], g22[mask]

    n_total = len(t)
    if n_total == 0:
        raise ValueError(f"No frames in {proj_path}")

    discard_n = compute_discard_n(n_total, discard_frac)
    if discard_n > 0:
        g11, g12, g21, g22 = g11[discard_n:], g12[discard_n:], g21[discard_n:], g22[discard_n:]

    n = len(g11)
    M11 = g11.mean()
    M22 = g22.mean()
    M12 = 0.5 * (g12.mean() + g21.mean())
    M = np.array([[M11, M12], [M12, M22]], dtype=float)
    return M, None, n


def FES_gradient(grid_point, discard_frac=0.0, tmin=None, tmax=None, kappa=1000.0):
    phi_deg, psi_deg, phi, psi = grid_point
    dir_name = f"phi_{phi_deg}_psi_{psi_deg}"
    colvar_file = f"{dir_name}/COLVAR_restrained"

    colvar = np.loadtxt(colvar_file, comments="#")
    if colvar.ndim == 1:
        colvar = colvar.reshape(1, -1)
    t = colvar[:, 0]
    phi_r = colvar[:, 1]
    psi_r = colvar[:, 2]

    mask = np.ones_like(t, dtype=bool)
    if tmin is not None:
        mask &= t >= tmin
    if tmax is not None:
        mask &= t <= tmax
    t, phi_r, psi_r = t[mask], phi_r[mask], psi_r[mask]

    n_total = len(t)
    if n_total == 0:
        raise ValueError(f"No frames in {colvar_file}")

    discard_n = compute_discard_n(n_total, discard_frac)
    if discard_n > 0:
        phi_r, psi_r = phi_r[discard_n:], psi_r[discard_n:]

    dphi = (phi_r - phi + np.pi) % (2.0 * np.pi) - np.pi
    dpsi = (psi_r - psi + np.pi) % (2.0 * np.pi) - np.pi

    grad_phi = -kappa * np.mean(dphi)
    grad_psi = -kappa * np.mean(dpsi)
    return grad_phi, grad_psi


def FES_metric(grid_point, discard_frac=0.0, tmin=None, tmax=None):
    phi_deg, psi_deg, phi, psi = grid_point
    dir_name = f"phi_{phi_deg}_psi_{psi_deg}"
    proj_file = os.path.join(dir_name, "PROJ")
    if not os.path.exists(proj_file):
        raise FileNotFoundError(f"Cannot find {proj_file}")
    M, _, _ = metric_from_proj(proj_file, tmin=tmin, tmax=tmax, discard_frac=discard_frac)
    return M


def delete_backupfile():
    for file_path in glob.glob(os.path.join(".", "bck.*.PLUMED.OUT")):
        try:
            os.remove(file_path)
        except OSError as exc:
            print(f"Error deleting {file_path}: {exc}")


def delete_temporary_phi_directories():
    subprocess.run(
        'find . -type d -name "phi_*" -exec rm -rf {} +',
        shell=True, check=False,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )


def M_gradient(q, discard_frac=0.2, tmin=None, tmax=None):
    q = q.cpu().detach().numpy()
    phi_deg = q[:, 0] / deg_to_rad
    psi_deg = q[:, 1] / deg_to_rad
    g = np.column_stack((phi_deg, psi_deg, q))[0]

    restrained_simulation(g)
    delete_backupfile()

    grad_phi, grad_psi = FES_gradient(g, discard_frac=discard_frac,
                                      tmin=tmin, tmax=tmax, kappa=1000.0)
    F_orig = np.array([grad_phi, grad_psi], dtype=float)
    M = FES_metric(g, discard_frac=discard_frac, tmin=tmin, tmax=tmax)
    delete_temporary_phi_directories()
    return M, F_orig


def gradient_md_use(q, discard_frac=INIT_DISCARD_FRAC, tmin=None, tmax=None):
    """
    对整条内部路径逐点做 restrained MD，返回:
        M_all : (N,2,2)
        F_all : (N,2)   自由能梯度, kcal/mol
    """
    if not torch.is_tensor(q):
        raise TypeError(f"q must be a torch.Tensor, got {type(q)}")
    if q.ndim == 1:
        q = q.unsqueeze(0)
    if q.ndim != 2 or q.shape[1] != 2:
        raise ValueError(f"q must have shape (N,2), got {tuple(q.shape)}")

    q_cpu = q.detach().cpu()
    n_points = q_cpu.shape[0]

    M_list = []
    g_list = []
    t0 = time.perf_counter()

    for idx in range(n_points):
        q_i = q_cpu[idx:idx + 1]
        M_i, F_i = M_gradient(q_i, discard_frac=discard_frac, tmin=tmin, tmax=tmax)
        M_i = np.asarray(M_i, dtype=float).reshape(2, 2)
        F_i = np.asarray(F_i, dtype=float).reshape(2,) / 4.184  # kJ/mol -> kcal/mol
        M_list.append(M_i)
        g_list.append(F_i)
        print(f"  [init MD] image {idx + 1:>2d}/{n_points} done")

    elapsed = time.perf_counter() - t0
    print(f"serial wall time = {elapsed:.3f} s  ({n_points} points)")

    M_all = np.stack(M_list, axis=0)
    g_all = np.stack(g_list, axis=0)
    return M_all, g_all


# ============================================================
# 与绘图 cell 完全一致的指标计算（numpy 版）
# ============================================================
def central_difference_tangent(full_path):
    return full_path[2:] - full_path[:-2]


def compute_G_projectors(tangent, M_all):
    tangent = np.asarray(tangent, dtype=float)
    M_all = np.asarray(M_all, dtype=float)
    n_internal = tangent.shape[0]
    G_all = np.linalg.inv(M_all)
    P_perp_all = np.zeros((n_internal, 2, 2))
    identity = np.eye(2)
    for i in range(n_internal):
        v_i = tangent[i]
        G_i = G_all[i]
        norm_G_sq = float(v_i @ G_i @ v_i)
        if not np.isfinite(norm_G_sq) or norm_G_sq <= 1e-30:
            raise FloatingPointError(f"Near-zero G tangent norm at image {i}")
        tau_G = v_i / np.sqrt(norm_G_sq)
        G_tau_G = G_i @ tau_G
        denom = float(tau_G @ G_tau_G)
        if not np.isfinite(denom) or denom <= 1e-30:
            raise FloatingPointError(f"Invalid projector denominator at image {i}")
        P_parallel = np.outer(tau_G, G_tau_G) / denom
        P_perp_all[i] = identity - P_parallel
    return G_all, P_perp_all


def local_G_arclength_weights(full_path, G_all):
    path_internal = full_path[1:-1]
    seg_minus = path_internal - full_path[:-2]
    seg_plus = full_path[2:] - path_internal
    length_minus_sq = np.einsum("ni,nij,nj->n", seg_minus, G_all, seg_minus)
    length_plus_sq = np.einsum("ni,nij,nj->n", seg_plus, G_all, seg_plus)
    return 0.5 * (np.sqrt(np.maximum(length_minus_sq, 0.0)) +
                  np.sqrt(np.maximum(length_plus_sq, 0.0)))


def local_euclidean_arclength_weights(full_path):
    path_internal = full_path[1:-1]
    seg_minus = path_internal - full_path[:-2]
    seg_plus = full_path[2:] - path_internal
    return 0.5 * (np.linalg.norm(seg_minus, axis=1) + np.linalg.norm(seg_plus, axis=1))


def compute_arclength_residual(full_path):
    n_full = full_path.shape[0]
    ds = 1.0 / (n_full - 1)
    phi_prime = (full_path[2:] - full_path[:-2]) / (2.0 * ds)
    phi_second = (full_path[2:] - 2.0 * full_path[1:-1] + full_path[:-2]) / (ds ** 2)
    inner = np.einsum("ni,ni->n", phi_prime, phi_second)
    speed = np.linalg.norm(phi_prime, axis=1)
    integrand = inner ** 2 * speed
    return float(np.sum(integrand) * ds)


def recover_free_energy_profile(full_path, F_internal):
    n_internal = full_path.shape[0] - 2
    segments = full_path[1:] - full_path[:-1]
    seg_grad = np.zeros((n_internal + 1, 2))
    seg_grad[0] = F_internal[0]
    if n_internal > 1:
        seg_grad[1:n_internal] = 0.5 * (F_internal[:-1] + F_internal[1:])
    seg_grad[n_internal] = F_internal[-1]
    increments = np.einsum("ni,ni->n", seg_grad, segments)
    profile = np.zeros(n_internal + 2)
    profile[1:] = np.cumsum(increments)
    profile -= profile[0]
    barrier = float(np.max(profile) - profile[0])
    return profile, barrier, float(profile[-1] - profile[0])


def compute_all_path_metrics(full_path, M_all, F_all):
    tangent = central_difference_tangent(full_path)
    G_all, P_perp_G_all = compute_G_projectors(tangent, M_all)

    # ---- panel (a): G 度量版, 作用在 M@F 上 ----
    Mg_all   = np.einsum("nij,nj->ni", M_all, F_all)
    perpG_Mg = np.einsum("nij,nj->ni", P_perp_G_all, Mg_all)
    residual_G_sq = np.maximum(
        np.einsum("ni,nij,nj->n", perpG_Mg, G_all, perpG_Mg), 0.0)
    weights_G = local_G_arclength_weights(full_path, G_all)
    I_perp_G = float(np.sum(residual_G_sq * weights_G))

    # ---- panel (b): 欧氏版, 直接对 F 做欧氏正交投影 ----
    tau_norm = np.linalg.norm(tangent, axis=1, keepdims=True)
    tau_E = tangent / tau_norm                              # (n,2)
    # F_perp = F - (F·τ̂) τ̂
    F_dot_tau = np.einsum("ni,ni->n", F_all, tau_E)         # (n,)
    F_perp = F_all - F_dot_tau[:, None] * tau_E             # (n,2)
    residual_E_sq = np.sum(F_perp ** 2, axis=1)
    I_perp_E = float(np.sum(residual_E_sq *
                            local_euclidean_arclength_weights(full_path)))

    profile, barrier, final_dF = recover_free_energy_profile(full_path, F_all)
    I_arc = compute_arclength_residual(full_path)

    return {
        "I_perp_G": I_perp_G,
        "I_perp_E": I_perp_E,
        "I_arc": I_arc,
        "Delta_F_barrier": barrier,
        "Delta_F_final": final_dF,
        "free_energy_profile": profile,
    }


# ============================================================
# 主流程：构造初始路径 -> 一次 MD -> 计算指标 -> 保存
# ============================================================
def build_initial_internal_path():
    """C1 -> C2 的线性插值内部 image，(N,2)。"""
    t = np.linspace(0.0, 1.0, N + 2)[1:-1]
    path = (1.0 - t)[:, None] * C1[None, :] + t[:, None] * C2[None, :]
    return path


def main():
    print("=" * 70)
    print("初始化公共路径并保存初始数据（无任何优化）")
    print("=" * 70)

    phi_init = build_initial_internal_path()          # (N,2)
    full_path = np.vstack([C1[None, :], phi_init, C2[None, :]])  # (N+2,2)

    print(f"内部 image 数: {N}")
    print(f"起点 C1 = {C1}")
    print(f"终点 C2 = {C2}")
    print("开始对初始路径做一次逐点 restrained MD 评估 ...")

    q = torch.tensor(phi_init, dtype=torch.float64)
    M_all, F_all = gradient_md_use(q, discard_frac=INIT_DISCARD_FRAC)

    print("MD 评估完成，计算初始指标 ...")
    metrics = compute_all_path_metrics(full_path, M_all, F_all)

    print("-" * 70)
    print("初始路径指标：")
    print(f"  I_perp_G      = {metrics['I_perp_G']:.6e}")
    print(f"  I_perp_E      = {metrics['I_perp_E']:.6e}")
    print(f"  I_arc         = {metrics['I_arc']:.6e}")
    print(f"  Delta_F_barrier = {metrics['Delta_F_barrier']:.6f} kcal/mol")
    print(f"  Delta_F_final   = {metrics['Delta_F_final']:.6f} kcal/mol")
    print("-" * 70)

    # count = 0 表示初始路径，尚未进行任何优化步
    np.savez(
        OUTPUT_NPZ,
        # 路径与原始 oracle 数据
        C1=C1,
        C2=C2,
        N=np.asarray(N),
        phi_init=phi_init,
        full_path_init=full_path,
        M_init=M_all,
        F_init=F_all,          # kcal/mol
        arc_init=np.linspace(0.0, 1.0, N + 2)[1:-1],
        discard_frac=np.asarray(INIT_DISCARD_FRAC),
        # 与绘图 cell 一致的初始指标
        count=np.asarray(0.0),
        I_perp_G=np.asarray(metrics["I_perp_G"]),
        I_perp_E=np.asarray(metrics["I_perp_E"]),
        I_arc=np.asarray(metrics["I_arc"]),
        Delta_F_barrier=np.asarray(metrics["Delta_F_barrier"]),
        Delta_F_final=np.asarray(metrics["Delta_F_final"]),
        free_energy_profile=metrics["free_energy_profile"],
    )

    print(f"[SAVED] 初始公共数据已保存：\n  {OUTPUT_NPZ}")
    print("=" * 70)


if __name__ == "__main__":
    main()

初始化公共路径并保存初始数据（无任何优化）
内部 image 数: 20
起点 C1 = [-2.60866606  2.77661136]
终点 C2 = [-1.3304716   0.99118795]
开始对初始路径做一次逐点 restrained MD 评估 ...
  [init MD] image  1/20 done
  [init MD] image  2/20 done
  [init MD] image  3/20 done
  [init MD] image  4/20 done
  [init MD] image  5/20 done
  [init MD] image  6/20 done
  [init MD] image  7/20 done
  [init MD] image  8/20 done
  [init MD] image  9/20 done
  [init MD] image 10/20 done
  [init MD] image 11/20 done
  [init MD] image 12/20 done
  [init MD] image 13/20 done
  [init MD] image 14/20 done
  [init MD] image 15/20 done
  [init MD] image 16/20 done
  [init MD] image 17/20 done
  [init MD] image 18/20 done
  [init MD] image 19/20 done
  [init MD] image 20/20 done
serial wall time = 22.022 s  (20 points)
MD 评估完成，计算初始指标 ...
----------------------------------------------------------------------
初始路径指标：
  I_perp_G      = 1.902623e+02
  I_perp_E      = 1.006851e+01
  I_arc         = 2.146162e-25
  Delta_F_barrier = 2.400002 kcal/mol
  Delta_F_f